# Exploratory Data Analysis (EDA)

Pada notebook ini, kita akan menggabungkan semua teknik yang telah dipelajari dalam proses EDA menyeluruh. Dataset `titanic_engineered.csv` akan diperkaya lebih lanjut dan disimpan sebagai `titanic_eda_final.csv`.

## Langkah 1: Memahami Struktur Dataset

In [ ]:
import pandas as pd

df = pd.read_csv("titanic_engineered.csv")
print(f"Dimensi: {df.shape}")
print(f"Kolom: {df.columns.tolist()}")
df.info()

## Langkah 2: Memeriksa Missing Values

In [ ]:
missing = df.isnull().sum()
print(missing[missing > 0])

# Tangani missing age_group
df["age_group"] = df["age_group"].fillna("Anak")
print(f"\nTotal missing setelah handling: {df.isnull().sum().sum()}")

## Langkah 3: Statistik Deskriptif

In [ ]:
df.describe().round(2)

In [ ]:
for kolom in ["sex", "embarked", "age_group", "fare_category", "title"]:
    print(f"\n--- {kolom} ---")
    print(df[kolom].value_counts())

## Langkah 4: Analisis Keselamatan per Kategori

In [ ]:
# Per jenis kelamin
print("=== Keselamatan per Sex ===")
print(df.groupby("sex").agg(
    jumlah=("survived", "count"),
    selamat=("survived", "sum"),
    tingkat_selamat=("survived", "mean")
).round(4))

In [ ]:
# Per kelas
print("=== Keselamatan per Kelas ===")
print(df.groupby("pclass").agg(
    jumlah=("survived", "count"),
    selamat=("survived", "sum"),
    tingkat_selamat=("survived", "mean")
).round(4))

In [ ]:
# Pivot: Kelas x Sex
pivot = df.pivot_table(values="survived", index="pclass", columns="sex", aggfunc=["mean", "count"]).round(4)
print(pivot)

In [ ]:
# Per title
print(df.groupby("title")["survived"].agg(["count", "sum", "mean"]).round(4).sort_values("mean", ascending=False))

## Langkah 5: Analisis Korelasi

In [ ]:
kolom_numerik = ["survived", "pclass", "age", "sibsp", "parch", "fare", "family_size", "is_alone"]
korelasi_survived = df[kolom_numerik].corr()["survived"].drop("survived").sort_values(ascending=False)
print(korelasi_survived.round(4))

## Langkah 6: Feature Engineering Tambahan

In [ ]:
# Survival Priority
def tentukan_prioritas(row):
    skor = 0
    if row["sex"] == "female": skor += 2
    if row["age"] < 13: skor += 2
    if row["pclass"] == 1: skor += 1
    elif row["pclass"] == 2: skor += 0.5
    if skor >= 3: return "Tinggi"
    elif skor >= 1.5: return "Sedang"
    else: return "Rendah"

df["survival_priority"] = df.apply(tentukan_prioritas, axis=1)
print(df.groupby("survival_priority")["survived"].mean().round(4))

In [ ]:
# Fare per person
df["fare_per_person"] = df["fare"] / df["family_size"]
print(df[["fare", "family_size", "fare_per_person"]].head())

## Langkah 7: Ringkasan Temuan

In [ ]:
print("=" * 50)
print("RINGKASAN EDA DATASET TITANIC")
print("=" * 50)
print(f"Total penumpang: {len(df)}")
print(f"Selamat: {df['survived'].sum()} ({df['survived'].mean()*100:.1f}%)")
print(f"Tidak selamat: {(df['survived']==0).sum()} ({(1-df['survived'].mean())*100:.1f}%)")
print(f"\nFAKTOR KESELAMATAN UTAMA:")
sr = df.groupby("sex")["survived"].mean()
print(f"  Sex: Female {sr['female']*100:.1f}% vs Male {sr['male']*100:.1f}%")
cr = df.groupby("pclass")["survived"].mean()
print(f"  Kelas: 1 ({cr[1]*100:.1f}%), 2 ({cr[2]*100:.1f}%), 3 ({cr[3]*100:.1f}%)")
print(f"\nEVOLUSI DATASET:")
print(f"  titanic_raw.csv       -> 891 x 12 (mentah)")
print(f"  titanic_cleaned.csv   -> 891 x 10 (bersih)")
print(f"  titanic_engineered.csv-> 891 x 13 (fitur baru)")
print(f"  titanic_eda_final.csv -> 891 x {df.shape[1]} (final)")

## Langkah 8: Menyimpan Dataset Akhir

In [ ]:
df.to_csv("titanic_eda_final.csv", index=False)
print(f"Dataset disimpan sebagai titanic_eda_final.csv")
print(f"Dimensi: {df.shape}")
print(f"Kolom: {df.columns.tolist()}")